# Notebook da Camada Bronze

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"

# Dicionário entre tabela e sua respectiva codificação para ingestão na camada Bronze a partir dos CSVs
tabelas = {"chamados_hora": "ISO8859-1", 
           "base_atendentes": "UTF-8",
           "base_motivos": "UTF-8",
           "canais": "UTF-8",
           "chamados": "UTF-8", 
           "clientes": "ISO8859-1",
           "custos": "UTF-8",
           "pesquisa_satisfacao": "UTF-8"}



In [0]:
# TRUNCATE TABLE caso desejado
# for t,_ in tabelas.items():
#     spark.sql(f"TRUNCATE TABLE {bronze_schema}.{t}")
#     print(f"Tabela {bronze_schema}.{t} truncada com sucesso.")

In [0]:
# Loop de criação das tabelas
for tabela, codificacao in tabelas.items():
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .option("encoding", codificacao)
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

In [0]:
#numero de rows em cada tabela
tabelas = spark.catalog.listTables(bronze_schema)
for t in tabelas:
  print(f"{t.name}: {spark.table(f'{bronze_schema}.{t.name}').count()} linhas")

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serve de apoio para as transformações feitas na camada silver)

for t in tabelas:
    display(spark.table(f'{bronze_schema}.{t.name}'))